# W11A · 7장 보강 · 실습 A · AE의 숫자와 관측 마스크

**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026-11-10**

다운로드 없는 독자 4×4 표로 순전파·관측 오차를 계산하고 같은 고정 모수의Keras 결과와 대조합니다. 마지막에는 누적 앱의 반환 표와CSV를 수정합니다.

Colab에서 자신의 사본을 저장하고 첫 셀부터 실행하세요. 설치 후 재시작 안내가 나오면 런타임을 재시작한 뒤 다시 처음부터 실행합니다. 개인 파일 업로드·API키·유료GPU가 필요하지 않습니다. 이 노트북은 독립적으로 입력을 준비합니다.

`None`은 자신의 답을 적는 자리이며 기본 계산 변수와 분리되어 있습니다. 빈 응답이 있어도 예제 실행은 계속되지만,이는 연습을 완료했다는 뜻이 아닙니다. 예상 → 직접 계산 → 실행 → 이유 한 문장 순서로 기록하세요.


In [ ]:
from __future__ import annotations
import os, sys, subprocess
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_VERSION = '2026-fall-w11a'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        f'luna-recommender-course[apps,neural] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.31.0'], check=True)
    for package, wanted in [('tensorflow','2.21.0'), ('keras','3.15.1')]:
        if package in sys.modules and getattr(sys.modules[package], '__version__', None) != wanted:
            raise RuntimeError('설치가 갱신되었습니다. 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요.')
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from IPython.display import display
from luna_recsys.autoencoder_recommenders import (
    toy_ratings, fixed_ae_forward, masked_metrics, loss_comparison,
    build_fixed_autoencoder, prepare_autoencoder_data, oriented_arrays,
    fit_autoencoder, predict_autoencoder, evaluate_autoencoder, recommend_autoencoder,
)
from luna_recsys.autoencoder_explainer import (
    ae_view, export_ae_csv, build_autoencoder_explainer,
)
from luna_recsys.datasets import prepare_movielens
print('Python',sys.version.split()[0], 'NumPy',np.__version__, 'TensorFlow',tf.__version__, 'Keras',keras.__version__)
print('수업 ref:',COURSE_VERSION,'CPU 실행; 학습 자료 모드를 결과와 함께 확인합니다.')


## W11A 보강: 함수의 내부를 본문에서 실행하기

기존 문제 번호·평점표·분할을 유지합니다. 이번에는 아래 실제 함수를 읽고 한 조건을 수정한 뒤 중간값과 결과를 설명합니다. `loss_fn`과 `step_fn`으로 후속 학습에 이 정의를 전달합니다.


### 本문 실행 정의 · `fixed_ae_forward`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: bebdd7e1f667ac2bfdb3f05a31f830a99077141c326790f3e748a2ab3d1a59ff
def fixed_ae_forward(raw: Any = None, *, mean: float = 3.) -> dict:
    """주어진 고정 모수4→2(tanh)→4(linear)의 NumPy 순전파.

    raw: 길이4의1–5 평점/NaN, 기본toy사용자A. mean은 주어진 훈련 평균이며
    이 함수에서 재적합하지 않는다. 관측 여부를 먼저 보존하고 빈칸만0으로
    채운다. 반환 배열은 새 사본이다. 학습/파일 쓰기 없음.
    fixed_ae_forward()['pre'] -> [1,-1].
    """
    ratings = toy_ratings()[0] if raw is None else np.asarray(raw, dtype=float).copy()
    observed = np.isfinite(ratings)
    if (ratings.shape != (4,) or np.isinf(ratings).any() or not np.isfinite(mean)
            or ((ratings[observed] < 1) | (ratings[observed] > 5)).any()):
        raise ValueError('길이4의1–5 평점/NaN과 유한한 평균이 필요합니다.')
    x = np.where(observed, ratings - mean, 0.)
    encoder = np.array([[.5, 0.], [0., .5], [.5, .5], [0., .5]])
    decoder = np.array([[1., .5, 0., -1.], [0., .5, 1., 0.]])
    pre = x @ encoder  # 두 bias는 설명용으로0이다.
    latent = np.tanh(pre)
    centered = latent @ decoder
    return {'raw': ratings, 'observed': observed, 'mean': float(mean),
            'input': x, 'encoder': encoder, 'encoder_bias': np.zeros(2),
            'pre': pre, 'latent': latent, 'decoder': decoder,
            'decoder_bias': np.zeros(4), 'centered_prediction': centered,
            'rating_prediction': centered + mean}


### 本문 실행 정의 · `observed_reconstruction_loss`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: 57aa25288e501a3b9481a24cc25ee4113e8a6b84ff7d5f412179a82323540721
def observed_reconstruction_loss(target: Any, prediction: Any, observed: Any, *, loss_kind: str = 'mse'):
    """Same-shaped tensors -> observed MSE (or stabilized RMSE); no updates.

    Explicit observation mask retains centered observed zeros. An empty mask
    is an error. L1 is added by the training step, not by this reconstruction loss.
    """
    import tensorflow as tf
    mask = tf.cast(observed, tf.float32)
    tf.debugging.assert_equal(tf.shape(target), tf.shape(prediction))
    tf.debugging.assert_equal(tf.shape(target), tf.shape(mask))
    tf.debugging.assert_positive(tf.reduce_sum(mask))
    if loss_kind not in {'mse', 'rmse'}:
        raise ValueError('loss_kind must be mse or rmse.')
    delta = tf.where(tf.cast(mask, tf.bool), target - prediction, tf.zeros_like(prediction))
    mse = tf.reduce_sum(tf.square(delta)) / tf.reduce_sum(mask)
    return mse if loss_kind == 'mse' else tf.sqrt(mse + 1e-8)


### 本문 실행 정의 · `autoencoder_train_step`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: 826ef138cafbfc08b7efdc7d2c1f1a329257de9788f6f330cf1457a6813b027a
def autoencoder_train_step(model: Any, optimizer: Any, target: Any, observed: Any, *,
                           loss_kind: str = 'mse', loss_fn=None) -> dict:
    """One real update. Optional loss_fn is the notebook's actual editable loss.

    Changes model weights and optimizer state; returns objective and gradient norm.
    Default loss preserves the previous observed-MSE/RMSE+L1 calculation.
    """
    import tensorflow as tf
    loss_fn = loss_fn or observed_reconstruction_loss
    with tf.GradientTape() as tape:
        pred = model(target, training=True)
        reconstruction = loss_fn(target, pred, observed, loss_kind=loss_kind)
        objective = reconstruction + (tf.add_n(model.losses) if model.losses else 0.)
    gradients = tape.gradient(objective, model.trainable_variables)
    if not bool(tf.math.is_finite(objective)) or any(
            g is None or not bool(tf.reduce_all(tf.math.is_finite(g))) for g in gradients):
        raise FloatingPointError('학습 목적함수 또는 기울기가 유한하지 않습니다.')
    norm = float(tf.linalg.global_norm(gradients))
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return {'objective': float(objective), 'gradient_norm': norm}


In [ ]:
module_ae_view = ae_view
def ae_view(r1=5.,r2=3.,r3=3.,r4=1.,o1=True,o2=False,o3=True,o4=True):
    return module_ae_view(r1,r2,r3,r4,o1,o2,o3,o4,forward_fn=fixed_ae_forward)


## 재사용 API와 배열을 먼저 읽기

`fixed_ae_forward(raw=None,mean=3)`는 작은 고정 모수의 계산 사전을 반환하며 학습하지 않습니다. `masked_metrics(y,p,m)`는 같은 크기의 배열에서 관측된 항만 평가합니다. `build_fixed_autoencoder(forward_fn=fixed_ae_forward)`는 같은 가중치의Keras 모형을 만듭니다. 입력 `(B,4)`,잠재표현 `(B,2)`,출력 `(B,4)`입니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w10b-r1/course/notion/assets/w10b/07-zero-mask.png" alt="두 종류의0과 관측 마스크" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w10b-r1/course/notion/assets/w10b/07-zero-mask.png)

독자 수계산 도해입니다. 빈칸의0과평균을 뺀 관측값0을 마지막 행의 마스크로 구분하세요.


In [ ]:
ratings = toy_ratings()
trace = fixed_ae_forward()
display(pd.DataFrame(ratings, index=list('ABCD'), columns=list('PQRS')))
print('설명용 관측 평균:', np.nanmean(ratings))
print('A 입력:',trace['input'],'관측:',trace['observed'])
print('가중합:',trace['pre'],'잠재값:',trace['latent'])
display(pd.DataFrame({'movie':list('PQRS'),'centered_prediction':trace['centered_prediction'],
                      'rating_prediction':trace['rating_prediction']}))


## L01 · 관측과 중심화

사용자B `[4,2,빈칸,3]`,평균3입니다. 입력과 관측 마스크를 먼저 적고,두 종류의0을 설명하세요.


In [ ]:
my_input_B = None
my_mask_B = None
my_zero_explanation = None


## L02 · 다른 입력의 순전파

B의 두 가중합과 tanh 뒤 잠재값,중심화 출력,평균을 복원한 평점을 구하세요. 가중치는 강의E02와같고 편향은0입니다.


In [ ]:
my_pre_B = None
my_latent_B = None
my_prediction_B = None


## 고정 모수의NumPy와Keras를 대조하기

아래는 강의E02의A 예제를 검산합니다. 모형을 만들고 예측할 뿐 학습하지 않습니다. 모수 수는22개입니다. 다음 계산 전에 같은 값이 나올지 예상하세요.

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w10b-r1/course/notion/assets/w10b/05-dense-tanh.png" alt="Dense의 곱셈과tanh" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w10b-r1/course/notion/assets/w10b/05-dense-tanh.png)

독자 고정 모수 예제의 한 노드 계산입니다.


In [ ]:
fixed_model = build_fixed_autoencoder(forward_fn=fixed_ae_forward)
fixed_output = fixed_model(trace['input'][None,:], training=False).numpy()[0]
np.testing.assert_allclose(fixed_output, trace['centered_prediction'], atol=1e-6)
print('Keras 중심화 출력:',fixed_output,'모수 수:',fixed_model.count_params())


## 실제 갱신 한 번을 따로 관찰하기

위 고정 모형은 검산용으로 보존합니다. 아래 별도 모형에서 본문의 관측 손실과 학습 함수를 사용합니다. `weight_delta`는 실제 모수 변화이며, 단순히 함수를 표시한 결과가 아닙니다.


In [ ]:
update_model = build_fixed_autoencoder(forward_fn=fixed_ae_forward)
update_x = tf.constant(trace['input'][None,:],dtype=tf.float32)
update_mask = tf.constant(trace['observed'][None,:],dtype=tf.float32)
before_update = [w.copy() for w in update_model.get_weights()]
update_result = autoencoder_train_step(update_model,keras.optimizers.Adamax(.001),update_x,update_mask,loss_fn=observed_reconstruction_loss)
weight_delta = sum(float(np.sum(abs(a-b))) for a,b in zip(before_update,update_model.get_weights()))
print('갱신:',update_result,'모수 변화 합:',weight_delta)
assert weight_delta>0


## R7A · 직접 설명하고 수정하기

본문의 `fixed_ae_forward`에서 인코더 가중치 하나를 바꾼 뒤 입력은 유지하세요. 잠재값·출력·앱 계산표가 어떻게 바뀌었는지 기록하고, 순전파 수정과 학습 갱신의 차이를 설명하세요.


In [ ]:
my_r7a = None


## L03 · 평균 복원과모수 수

평균3.5와중심화 예측 `[.4,-.2,.3,-.5]`를 복원하세요. `4→2→4`의모수 수와배치 100의출력 shape도 적으세요.


In [ ]:
my_restored_ratings = None
my_parameter_count = None
my_batch_shape = None


## 같은 예측에서오차 계산 대상을비교하기

강의E04의주어진예측 `[1,.4,2,-1]`로 오차만 분리해 봅니다. 앞의고정 모형출력과다른 주어진수치입니다. 실제관측 수와0이아닌정답수를먼저예상하세요.


In [ ]:
example_target = np.array([2.,0.,0.,-2.])
example_prediction = np.array([1.,.4,2.,-1.])
example_observed = np.array([True,False,True,True])
example_losses = loss_comparison(example_target,example_prediction,example_observed)
display(example_losses)


## L04 · 빈칸의예측만바꾸기

주어진예측을 `[1,4,2,-1]`로바꾸어세RMSE를직접계산하세요. 전체 칸RMSE가항상더작은지도설명하세요.


In [ ]:
my_squared_errors = None
my_observed_rmse = None
my_all_rmse = None
my_nonzero_rmse = None


## L05 · 관측값0의반례

원래 평점 `[3,4,빈칸,2]`,평균3,주어진중심화 예측 `[2,0,.5,0]`입니다. 관측 마스크로고치는방법과두RMSE차이를적으세요.


In [ ]:
my_observed_L05 = None
my_fixed_selection_code = None
my_zero_row_explanation = None


## L06 · 한 가중치의SGD

강의E06의단일디코더가중치에서학습률을.2로바꾸세요. 새 가중치·예측·MSE를구하고학습률10과비교하세요. 다른모수는고정,정규화0입니다.


In [ ]:
my_decoder_gradient = None
my_weight_after = None
my_mse_after = None
my_optimizer_explanation = None


## L08 · 앱표시 함수와CSV수정

P의평점5를4로바꿀때결과를예상하세요. 이어ae_view를감싸는자신의함수에서관측된 칸의절대오차 열을추가하고,미관측 칸의표시규칙을정하세요. 같은5개반환항목과기본호출을유지합니다.


In [ ]:
my_ae_view = None  # 완성한함수를여기에연결합니다. 기본None은기존계산을사용합니다.
my_before_after_explanation = None


## 누적 앱 실행과다운로드

기존FM/MF/MLP복습과AE기능을함께사용합니다. 고정 모수순전파와별도합성모형학습을구별하세요. 자신의함수를연결했다면CSV에도같은열이있는지확인합니다. 공유주소는런타임이켜져있는동안유효합니다.


In [ ]:
view_callback = my_ae_view if callable(my_ae_view) else ae_view
demo = build_autoencoder_explainer(view_fn=view_callback)
if IN_COLAB:
    demo.queue(default_concurrency_limit=1).launch(share=True,inline=False,prevent_thread_lock=True)
else:
    print('로컬검증에서는앱 객체와콜백을검사합니다. 실제브라우저조작은별도검증합니다.')


In [ ]:
np.testing.assert_allclose(fixed_output,trace['centered_prediction'],atol=1e-6)
assert fixed_model.count_params()==22
assert example_losses.n.tolist()==[3,4,2]
np.testing.assert_allclose(example_losses.rmse,[np.sqrt(2),np.sqrt(1.54),1])
assert keras.layers.Masking().compute_mask(tf.zeros((1,4))).numpy().shape == (1,)
assert view_callback()[1].shape[0]==4
assert Path(export_ae_csv(view_fn=view_callback)).is_file()
print('W11A_7A_RUNTIME_PASS demo/API checks; learner responses are not completion claims')


## 점검 퀴즈

1. 사용자 기준 AE의 한 입력 샘플은 무엇이며, MF의 ID 입력과 어떻게 다른가요?
2. 입력이 4차원이고 잠재표현이 2차원일 때, 출력이 다시 4차원이어야 하는 이유는 무엇인가요?
3. 입력 `[2,0,0,−2]`와 E02의 가중치에서 두 잠재 노드의 가중합과 활성값은 각각 무엇인가요?
4. 원래 평점3, 훈련 평균3인 관측이 중심화 후 0이 되면, 오차 계산에 포함해야 하나요?
5. E04의 관측 제곱오차 합 6과 관측 수 3에서 MSE와 RMSE는 각각 얼마인가요?
6. 관측 MSE 2와 L1 벌점 .0105가 주어졌을 때, 최적화 목적함수와 순수 RMSE는 어떻게 다른가요?
7. 교재의 `validation_data=(train_data, test_data)`에서 모델 입력과 비교할 정답은 각각 무엇인가요?
8. 같은 관측 분할로 사용자 기준과 아이템 기준을 비교할 때, RMSE 외에 어떤 조건과 수를 함께 기록해야 하나요?


## 기억할 것과 참고자료

입력과 관측 여부, 순전파와 학습, 검증과 시험을 구별하세요. 함수의 배열 크기와 상태 변경은 API 길잡이에서 확인합니다. 예상·수정·실행·이유를 자신의 말로 기록하세요.

- 주교재 임일(2025), 7장 전체, pp.148–156, 연습 7-1.
- [강의노트와워크북](https://github.com/lunalab-ai/recommender/tree/2026-fall-w10b-r1/course/notion)
- [Keras Dense](https://keras.io/api/layers/core_layers/dense/)
- [Keras Masking](https://keras.io/api/layers/core_layers/masking/)
- [AutoRec 원논문](https://ssanner.github.io/papers/anu/www15.pdf)
- [MovieLens 이용조건](https://files.grouplens.org/datasets/movielens/ml-100k-README.txt)
